In [ ]:
"""
═══════════════════════════════════════════════════════════════════
Supplementary Figures 
═══════════════════════════════════════════════════════════════════
"""

import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib import cm
from scipy.stats import linregress, spearmanr

# ═════════════════════════════════════════════════════════════════
# CONFIG
# ═════════════════════════════════════════════════════════════════
SHP_PATH   = Path("shape/Flowlines_huc02_12_Brazos.shp")
RESULT_DIR = Path("outputs_flood_rl2")
FIG_DIR    = Path("figures_wrr_new"); FIG_DIR.mkdir(parents=True, exist_ok=True)

GCMS = ["MPI-ESM1-2-HR","ACCESS-CM2","EC-Earth3",
        "CNRM-ESM2-1","BCC-CSM2-MR","MRI-ESM2-0","NorESM2-MM"]
SCENARIOS     = [126, 245, 370, 585]
TARGET_BUDGET = 10.0

SSP_LABELS = {126:"SSP1-2.6", 245:"SSP2-4.5", 370:"SSP3-7.0", 585:"SSP5-8.5"}
SSP_COLORS = {126:"#2d8e3d", 245:"#2166AC", 370:"#d4820a", 585:"#c72e29"}

MM = 1/25.4; W_FULL = 190*MM

mpl.rcParams.update({
    "font.family":"Arial", "font.size":7,
    "axes.labelsize":7, "axes.titlesize":7, "axes.linewidth":0.5,
    "axes.spines.top":False, "axes.spines.right":False,
    "xtick.labelsize":6, "ytick.labelsize":6,
    "xtick.major.width":0.4, "ytick.major.width":0.4,
    "xtick.major.size":2.5, "ytick.major.size":2.5,
    "xtick.direction":"out", "ytick.direction":"out",
    "legend.fontsize":5.5, "legend.frameon":False,
    "legend.handlelength":1.2, "legend.handletextpad":0.4,
    "figure.dpi":200, "savefig.dpi":300,
    "savefig.bbox":"tight", "savefig.pad_inches":0.02,
    "pdf.fonttype":42, "ps.fonttype":42, "lines.linewidth":0.8,

})

# Shared style constants
RIBBON_ALPHA  = 0.12   # ribbon alpha
MARKER_SIZE  = 4.0    # marker size
MARKER_EDGE_WIDTH = 0.3    # marker edge width
GRID_ALPHA  = 0.08   # grid alpha
PANEL_LABEL_X  = -0.10  # panel label x
PANEL_LABEL_Y  = 1.05   # panel label y
NSE_TH = 0.5
RETURN_PERIODS = [2, 5, 10, 20, 50, 100]

def lgrid(ax):
    ax.grid(True, alpha=GRID_ALPHA, lw=0.3, zorder=0); ax.set_axisbelow(True)

def plabel(ax, lab):
    ax.text(PANEL_LABEL_X, PANEL_LABEL_Y, lab, transform=ax.transAxes,
            fontweight="bold", fontsize=9, va="top", ha="right")

def weighted_quantile(values, quantiles, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    m = np.isfinite(v) & np.isfinite(w) & (w > 0)
    v, w = v[m], w[m]
    if len(v) == 0: return np.full_like(quantiles, np.nan, dtype=float)
    s = np.argsort(v); v, w = v[s], w[s]
    cw = np.cumsum(w); cdf = (cw - 0.5*w)/cw[-1]
    return np.interp(quantiles, cdf, v)

def flow_weighted_median(x, w):
    return float(weighted_quantile(x, [0.5], w)[0])


# ═════════════════════════════════════════════════════════════════
# INTERPOLATION
# ═════════════════════════════════════════════════════════════════
def interpolate_to_budget(grp, target=TARGET_BUDGET):
    eff_by_r = grp.groupby("R_weight")["eff_total_L1"].sum()/1e9
    r_values, efforts = eff_by_r.index.values, eff_by_r.values
    if len(r_values) < 2:
        return grp[grp["R_weight"]==r_values[0]].copy()
    si = np.argsort(efforts); es, rs = efforts[si], r_values[si]
    if target<=es[0]:  return grp[grp["R_weight"]==rs[0]].copy()
    if target>=es[-1]: return grp[grp["R_weight"]==rs[-1]].copy()
    ia = np.searchsorted(es, target); ib = ia-1
    w = 0.5 if es[ia]==es[ib] else (target-es[ib])/(es[ia]-es[ib])
    dlo = grp[grp["R_weight"]==rs[ib]].set_index("reach_id")
    dhi = grp[grp["R_weight"]==rs[ia]].set_index("reach_id")
    common = dlo.index.intersection(dhi.index)
    dlo, dhi = dlo.loc[common], dhi.loc[common]
    out = dlo.copy()
    for c in dlo.select_dtypes(include=[np.number]).columns:
        if c in ("R_weight","scenario","reach_id"): continue
        out[c] = dlo[c]*(1-w)+dhi[c]*w
    return out.reset_index()


# ═════════════════════════════════════════════════════════════════
# DATA LOADING
# ═════════════════════════════════════════════════════════════════
print("Loading data …")
gdf_net = gpd.read_file(SHP_PATH)
gdf_net["COMID"] = gdf_net["COMID"].astype(int)

frames = []
for gcm in GCMS:
    for pat in [f"results_*_{gcm}.parquet", f"results_{gcm}.parquet"]:
        for fp in RESULT_DIR.glob(pat): frames.append(pd.read_parquet(fp))
df_all = pd.concat(frames, ignore_index=True)
df_all["reach_id"] = df_all["reach_id"].astype(int)
df_all["scenario"] = df_all["scenario"].astype(int)
if "hydro" not in df_all.columns: df_all["hydro"] = "VIC5"

print(f"  Interpolating to {TARGET_BUDGET} Gm³/yr …")
sel = []
for k, g in df_all.groupby(["gcm","scenario","hydro"]):
    sel.append(interpolate_to_budget(g))
df_sel = pd.concat(sel, ignore_index=True)
df_sel["scenario"] = df_sel["scenario"].astype(int)
df_sel["reach_id"] = df_sel["reach_id"].astype(int)

mc = [c for c in ["vol_hi_baseline","vol_hi_controlled","eff_total_L1",
                   "residual_ratio","mean_flow_ref"] if c in df_sel.columns]
df_ens = df_sel.groupby(["reach_id","scenario"])[mc].median().reset_index()
df_ens = df_ens.merge(gdf_net[["COMID","StreamOrde","TotDASqKM"]].rename(
    columns={"COMID":"reach_id"}), on="reach_id", how="left")
if "mean_flow_ref" in df_sel.columns and "mean_flow_ref" not in df_ens.columns:
    df_ens["mean_flow_ref"] = df_ens["reach_id"].map(
        df_sel.groupby("reach_id")["mean_flow_ref"].first())
df_ens["so"] = df_ens["StreamOrde"].astype(float).astype("Int64")

daily_frames = [pd.read_parquet(f) for f in sorted(RESULT_DIR.glob("daily_*.parquet"))]
df_daily = pd.concat(daily_frames, ignore_index=True) if daily_frames else None
if df_daily is not None:
    df_daily = df_daily[df_daily["budget_target"]==TARGET_BUDGET]
    df_daily["date"] = pd.to_datetime(df_daily["date"])
    df_daily["year"] = df_daily["date"].dt.year

def baseline_per_member(df):
    idx = df.groupby(["gcm","hydro","scenario"])["R_weight"].transform("min")==df["R_weight"]
    return df[idx].drop_duplicates(subset=["gcm","hydro","scenario","reach_id"])
df_base = baseline_per_member(df_all)

if "atten_fraction" not in df_base.columns and "eff_atten_L1" in df_base.columns:
    df_base["atten_fraction"] = df_base["eff_atten_L1"]/(df_base["eff_total_L1"]+1e-12)

print(f"  {df_ens['reach_id'].nunique():,} reaches\n")

In [ ]:
# %% ═════════════════════════════════════════════════════════════
# FIG S1_validation — Surrogate Fidelity During Extreme Flows (1×3)
#   (a) Per-reach RL10 bias histogram
#   (b) Basin-total RL bias by return period (jitter + ensemble median)
#   (c) RL10 bias by stream order (jitter + ensemble median)
#
# Uses pre-computed RL bias columns from outputs_flood_rl2/*.parquet
# (same data loaded in cell 0 as df_all / df_base)
# ═════════════════════════════════════════════════════════════════
print("FIG S1_validation — Surrogate Fidelity")

RPS = [2, 5, 10, 20, 50, 100]
NSE_TH_VAL = 0.5

# Filter to one R_weight per member (baseline)
df_val = df_base.copy()
if "nse_openloop" in df_val.columns:
    df_val = df_val[df_val["nse_openloop"] > NSE_TH_VAL]
print(f"  {len(df_val):,} reach-member rows (NSE > {NSE_TH_VAL})")

# Merge stream order
if "StreamOrde" not in df_val.columns:
    df_val = df_val.merge(
        gdf_net[["COMID","StreamOrde"]].rename(columns={"COMID":"reach_id"}),
        on="reach_id", how="left")
df_val["so"] = df_val["StreamOrde"].astype(float).astype("Int64")

np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(W_FULL, 65*MM),
                          gridspec_kw={"wspace": 0.38})

# ── (a) Per-reach RL10 bias histogram ────────────────────────────
ax = axes[0]
if "rl10_bias_openloop" in df_val.columns:
    bias_arr = pd.to_numeric(df_val["rl10_bias_openloop"], errors="coerce").dropna().values
    bias_arr = bias_arr[np.isfinite(bias_arr)]
    ax.hist(np.clip(bias_arr, -1, 1), bins=80, range=(-1, 1),
            color="#9ecae1", edgecolor="#4292c6", linewidth=0.3,
            density=True, zorder=2)
    med = np.median(bias_arr)
    ax.axvline(med, color="k", ls="--", lw=0.8, zorder=3)
    ax.axvline(0, color="#999", ls="-", lw=0.4, zorder=1)
    ymax = ax.get_ylim()[1]
    ax.text(med - 0.03, ymax * 0.92,
            f"median\n{med:.3f}", fontsize=5.5, ha="right", va="top")
    ax.set_xlim(-1, 1)
    print(f"  (a) RL10 bias median = {med:.4f}")
ax.set_xlabel(r"RL$_{10}$ relative bias")
ax.set_ylabel("Density")
lgrid(ax); plabel(ax, "a")

# ── (b) Basin-total RL bias by return period ─────────────────────
ax = axes[1]
member_cols = ["gcm"] + (["hydro"] if "hydro" in df_val.columns else [])

# Compute basin-total bias per member per return period
# Use flow-weighted sum: sum(rl_bias * mean_flow) / sum(mean_flow)
bias_rows = []
for keys, grp in df_val.groupby(member_cols + ["scenario"]):
    row = {}
    for T in RPS:
        col = f"rl{T}_bias_openloop"
        if col in grp.columns:
            vals = pd.to_numeric(grp[col], errors="coerce").dropna()
            if "mean_flow_ref" in grp.columns:
                w = grp.loc[vals.index, "mean_flow_ref"].fillna(1).values
                row[T] = np.average(vals.values, weights=np.abs(w) + 1e-12) * 100
            else:
                row[T] = vals.median() * 100
    if row:
        bias_rows.append(row)

if bias_rows:
    bias_mat = np.array([[r.get(rp, np.nan) for rp in RPS] for r in bias_rows])

    log_rps = np.log10(RPS)
    for i in range(bias_mat.shape[0]):
        jitter = np.random.uniform(-0.06, 0.06, len(RPS))
        x_jit = 10 ** (log_rps + jitter)
        ax.scatter(x_jit, bias_mat[i, :],
                   s=5, color="#9ecae1", alpha=0.35, zorder=1,
                   edgecolors="none")

    med_line = np.nanmedian(bias_mat, axis=0)
    ax.plot(RPS, med_line, "o-", color="#d62728",
            ms=5, lw=1.0, zorder=3, label="Ensemble median")

    ax.axhline(0, color="#999", ls="-", lw=0.4, zorder=0)
    ax.axhline(5, color="#999", ls=":", lw=0.5, zorder=0)
    ax.axhline(-5, color="#999", ls=":", lw=0.5, zorder=0)
    ax.set_ylim(-10, 10)
    ax.text(0.97, 0.05,
            f"n = {bias_mat.shape[0]} members",
            transform=ax.transAxes, fontsize=5, va="bottom", ha="right",
            color="#555")
    print(f"  (b) {bias_mat.shape[0]} members")

ax.set_xlabel("Return period (years)")
ax.set_ylabel("Basin-total RL bias (%)")
ax.set_xscale("log")
ax.set_xticks(RPS)
ax.set_xticklabels([str(r) for r in RPS])
ax.legend(loc="lower left", fontsize=5)
lgrid(ax); plabel(ax, "b")

# ── (c) RL10 bias by stream order ────────────────────────────────
ax = axes[2]
if "rl10_bias_openloop" in df_val.columns and "so" in df_val.columns:
    all_orders = sorted(df_val["so"].dropna().unique())
    all_orders = [o for o in all_orders if o > 0]

    # Per member per stream order: flow-weighted RL10 bias
    order_rows = []
    for keys, grp in df_val.groupby(member_cols + ["scenario"]):
        row = {}
        for order in all_orders:
            sub = grp[grp["so"] == order]
            vals = pd.to_numeric(sub["rl10_bias_openloop"], errors="coerce").dropna()
            if len(vals) > 0:
                if "mean_flow_ref" in sub.columns:
                    w = sub.loc[vals.index, "mean_flow_ref"].fillna(1).values
                    row[order] = np.average(vals.values, weights=np.abs(w) + 1e-12) * 100
                else:
                    row[order] = vals.median() * 100
        if row:
            order_rows.append(row)

    if order_rows:
        order_mat = np.array([[r.get(o, np.nan) for o in all_orders]
                               for r in order_rows])

        x_pos = np.arange(len(all_orders))
        for i in range(order_mat.shape[0]):
            jitter = np.random.uniform(-0.15, 0.15, len(all_orders))
            ax.scatter(x_pos + jitter, order_mat[i, :],
                       s=5, color="#9ecae1", alpha=0.35, zorder=1,
                       edgecolors="none")

        med_o = np.nanmedian(order_mat, axis=0)
        ax.plot(x_pos, med_o, "o-", color="#d62728",
                ms=5, lw=1.0, zorder=3, label="Ensemble median")

        ax.axhline(0, color="#999", ls="-", lw=0.4, zorder=0)
        ax.axhline(5, color="#999", ls=":", lw=0.5, zorder=0)
        ax.axhline(-5, color="#999", ls=":", lw=0.5, zorder=0)
        ax.set_xticks(x_pos)
        ax.set_xticklabels([str(int(o)) for o in all_orders])
        ax.legend(loc="upper right", fontsize=5)

ax.set_ylim(-10, 10)
ax.set_xlabel("Stream order")
ax.set_ylabel(r"RL$_{10}$ bias (%)")
lgrid(ax); plabel(ax, "c")

for ext in ("png", "pdf"):
    fig.savefig(FIG_DIR / f"figS1_validation.{ext}")
plt.show()
print("Saved figS1_validation")

In [ ]:
# %% FIG S2 — Surrogate Mass Fidelity (1×3)
# (a) Volume ratio  (b) δ consistency r+R²  (c) Per-reach δ median
# Reads mass_conservation_summary.csv (from si_mass_conservation.py)

MASS_CSV = Path("mass_conservation_summary.csv")

def jitter_strip(ax, x_pos, values, color, width=0.25, ms=3.5, alpha=0.5):
    jit = np.random.default_rng(42).uniform(-width/2, width/2, len(values))
    ax.scatter(x_pos + jit, values, s=ms, alpha=alpha, color=color,
               edgecolors="black", linewidths=0.15, zorder=3, rasterized=True)

def dot_whisker(ax, x_pos, values, color, lw=2.0, ms=7):
    med = np.median(values)
    q25, q75 = np.percentile(values, [25, 75])
    ax.plot([x_pos, x_pos], [q25, q75], color=color, lw=lw,
            solid_capstyle="round", zorder=4)
    ax.plot(x_pos, med, "o", color=color, ms=ms,
            markeredgecolor="black", markeredgewidth=0.5, zorder=5)
    return med

if MASS_CSV.exists():
    df_mass = pd.read_csv(MASS_CSV)
    print(f"FIG S5 — Loaded {len(df_mass)} members")
    scenarios_mass = sorted(df_mass["ssp"].unique())

    fig, axes = plt.subplots(1, 3, figsize=(W_FULL, 70*MM),
                              gridspec_kw={"wspace": 0.36})

    # ── (a) Volume ratio ────────────────────────────────────────
    ax = axes[0]
    ax.axhline(1.0, color="#777", ls="--", lw=0.6, zorder=1)
    for i, scn in enumerate(scenarios_mass):
        vals = df_mass[df_mass["ssp"]==scn]["vol_ratio"].dropna().values
        jitter_strip(ax, i, vals, SSP_COLORS[scn], ms=4, alpha=0.55)
        dot_whisker(ax, i, vals, SSP_COLORS[scn])
    all_vr = df_mass["vol_ratio"].dropna().values
    ax.text(0.97, 0.05, f"all: {np.median(all_vr):.3f}\n(n={len(all_vr)})",
            transform=ax.transAxes, fontsize=6.5, va="bottom", ha="right",
            bbox=dict(boxstyle="round,pad=0.15", fc="white",
                                    ec="#999", lw=0.3, alpha=0.85))
    ax.set_xticks(range(len(scenarios_mass)))
    ax.set_xticklabels([SSP_LABELS[s] for s in scenarios_mass], fontsize=5.5)
    ax.set_ylabel("Volume ratio\n(surrogate / reference)")
    ax.set_xlim(-0.5, len(scenarios_mass)-0.5)
    ax.set_ylim(0.95, 1.05)
    lgrid(ax); plabel(ax, "a")

    # ── (b) δ consistency: r + R² ───────────────────────────────
    ax = axes[1]
    ax.axhline(1.0, color="#777", ls="--", lw=0.6, zorder=1)

    offset = 0.14
    for i, scn in enumerate(scenarios_mass):
        sub = df_mass[df_mass["ssp"]==scn]
        c = SSP_COLORS[scn]
        for j, (col, marker, ms, label) in enumerate([
            ("delta_corr", "o", 6, "Correlation $r$"),
            ("delta_r2",   "s", 5.5, "$R^2$"),
        ]):
            vals = sub[col].dropna().values
            xpos = i + (j - 0.5) * offset * 2
            jitter_strip(ax, xpos, vals, c, width=0.12, ms=3, alpha=0.4)
            med = np.median(vals)
            q25, q75 = np.percentile(vals, [25, 75])
            ax.plot([xpos, xpos], [q25, q75], color=c, lw=1.5,
                    solid_capstyle="round", zorder=3,
                    alpha=0.7 if j == 1 else 1.0)
            ax.plot(xpos, med, marker, color=c, ms=ms,
                    markeredgecolor="black", markeredgewidth=0.5,
                    zorder=4, label=label if i == 0 else "")

    all_r  = df_mass["delta_corr"].dropna().values
    all_r2 = df_mass["delta_r2"].dropna().values
    ax.text(0.97, 0.05,
            f"$r$: {np.median(all_r):.3f}\n$R^2$: {np.median(all_r2):.3f}",
            transform=ax.transAxes, fontsize=6.5, va="bottom", ha="right",
            bbox=dict(boxstyle="round,pad=0.15", fc="white",
                                    ec="#999", lw=0.3, alpha=0.85))
    ax.set_xticks(range(len(scenarios_mass)))
    ax.set_xticklabels([SSP_LABELS[s] for s in scenarios_mass], fontsize=5.5)
    ax.set_ylabel(r"Basin-total $\delta$ consistency")
    ax.set_xlim(-0.5, len(scenarios_mass)-0.5)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc="lower left", fontsize=5, handlelength=1.0)
    lgrid(ax); plabel(ax, "b")

    # ── (c) Per-reach δ correlation median ──────────────────────
    ax = axes[2]
    ax.axhline(1.0, color="#777", ls="--", lw=0.6, zorder=1)
    for i, scn in enumerate(scenarios_mass):
        vals = df_mass[df_mass["ssp"]==scn]["reach_median"].dropna().values
        jitter_strip(ax, i, vals, SSP_COLORS[scn], ms=4, alpha=0.55)
        dot_whisker(ax, i, vals, SSP_COLORS[scn])
    all_rm = df_mass["reach_median"].dropna().values
    ax.text(0.97, 0.05, f"all: {np.median(all_rm):.3f}\n(n={len(all_rm)})",
            transform=ax.transAxes, fontsize=6.5, va="bottom", ha="right",
            bbox=dict(boxstyle="round,pad=0.15", fc="white",
                                    ec="#999", lw=0.3, alpha=0.85))
    ax.set_xticks(range(len(scenarios_mass)))
    ax.set_xticklabels([SSP_LABELS[s] for s in scenarios_mass], fontsize=5.5)
    ax.set_ylabel("Per-reach $\\delta$ correlation\n(median across reaches)")
    ax.set_xlim(-0.5, len(scenarios_mass)-0.5)
    ax.set_ylim(-0.05, 1.05)
    lgrid(ax); plabel(ax, "c")

    for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS2_mass_fidelity.{ext}")
    plt.show()
else:
    print(f"FIG S5 — SKIPPED: {MASS_CSV} not found")
    print("  Run: python si_mass_conservation.py --gcm all --hydro all --ssp all")

In [ ]:
# %% ═════════════════════════════════════════════════════════════
# FIG S3 — Effort-Discharge Scaling (1×2)
# ═════════════════════════════════════════════════════════════════
print("FIG S3 — Effort-Discharge Scaling")
sub = df_ens[df_ens["scenario"]==245].copy()
eff = sub["eff_total_L1"].to_numpy(float)
da  = sub["TotDASqKM"].to_numpy(float)
mf  = sub["mean_flow_ref"].to_numpy(float) if "mean_flow_ref" in sub.columns else None

fig, axes = plt.subplots(1, 2, figsize=(W_FULL, 70*MM),
                          gridspec_kw={"wspace":0.28})

panels = [
    (axes[0], mf,  "Blues",
     r"Mean annual flow $\overline{Q}$ (log$_{10}$ m$^3$ s$^{-1}$)", "a"),
    (axes[1], da,  "Greens",
     r"Drainage area (log$_{10}$ km$^2$)", "b"),
]

for ax, xv, cmap_name, xlab, lab in panels:
    if xv is None:
        ax.text(0.5,0.5,"Data not available",transform=ax.transAxes,ha="center")
        plabel(ax,lab); continue

    m = (eff>0)&(xv>0)&np.isfinite(eff)&np.isfinite(xv)
    lx, ly = np.log10(xv[m]), np.log10(eff[m])
    lr = linregress(lx, ly)

    ax.hexbin(lx, ly, gridsize=45, cmap=cmap_name, mincnt=1,
              linewidths=0, alpha=0.75, zorder=1)

    xf = np.array([lx.min(), lx.max()])
    ax.plot(xf, lr.slope*xf+lr.intercept, "--", color="#333",
            lw=1.3, zorder=5)

    ax.text(0.05, 0.95,
            f"slope = {lr.slope:.2f}\n$R^2$ = {lr.rvalue**2:.2f}",
            transform=ax.transAxes, fontsize=6.5, va="top", ha="left",
            color="#333",
            bbox=dict(boxstyle="round,pad=0.2", fc="white",
                      ec="#999", lw=0.4, alpha=0.92))

    ax.set_xlabel(xlab)
    ax.set_ylim(-1, 8)
    if lab=="a": ax.set_ylabel(r"Annual effort (log$_{10}$ m$^3$ yr$^{-1}$)")
    lgrid(ax); plabel(ax, lab)
    print(f"  ({lab}) R²={lr.rvalue**2:.3f}")

for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS3_regression.{ext}")
plt.show()


# %% ═════════════════════════════════════════════════════════════

In [ ]:
# FIG S4 — Robustness (1×3)
#   (a) Stream-order effort fraction — grouped bar
#   (b) Temporal late/early ratio — dot + whisker
#   (c) Ranking stability — grouped dot plot
# ═════════════════════════════════════════════════════════════════
print("FIG S4 — Robustness")
so_groups = [("1–2",[1,2]), ("3–4",[3,4]), ("5+",[5,6,7,8,9])]

fig, axes = plt.subplots(1, 3, figsize=(W_FULL, 70*MM),
                          gridspec_kw={"wspace":0.38})

# ── (a) Stream-order effort ──────────────────────────────────────
ax = axes[0]
x_pos = np.arange(len(so_groups))
bw = 0.18
for i, scn in enumerate(SCENARIOS):
    sub = df_ens[df_ens["scenario"]==scn]
    total = sub["eff_total_L1"].sum()
    fracs = [sub[sub["so"].isin(o)]["eff_total_L1"].sum()/total*100
             for _,o in so_groups]
    ax.bar(x_pos+(i-1.5)*bw, fracs, bw,
           color=SSP_COLORS[scn], alpha=0.75, edgecolor="black",
           linewidth=0.3, label=SSP_LABELS[scn])

ax.set_xticks(x_pos)
ax.set_xticklabels([g for g,_ in so_groups])
ax.set_xlabel("Stream order")
ax.set_ylabel("Share of basin effort (%)")
ax.set_ylim(0, 58)
ax.legend(fontsize=4.5, loc="upper left", ncol=2,
          handlelength=0.8, columnspacing=0.5)
lgrid(ax); plabel(ax,"a")

for scn in SCENARIOS:
    sub = df_ens[df_ens["scenario"]==scn]
    t = sub["eff_total_L1"].sum()
    o5 = sub[sub["so"].isin([5,6,7,8,9])]["eff_total_L1"].sum()/t*100
    print(f"  (a) {SSP_LABELS[scn]}: Order 5+ = {o5:.1f}%")

# ── (b) Temporal late/early ratio — dot + whisker ────────────────
ax = axes[1]
if df_daily is not None:
    df_daily["period"] = np.where(df_daily["year"]<2060, "early", "late")
    grp = ["scenario","gcm"] + (["hydro"] if "hydro" in df_daily.columns else [])
    merge_on = ["gcm"] + (["hydro"] if "hydro" in df_daily.columns else [])

    pb = (df_daily.groupby(grp+["period"])
          .agg(cl=("basin_exceed_controlled","sum")).reset_index())
    pb["cl_Gm3"] = pb["cl"]/1e9

    for i, scn in enumerate(SCENARIOS):
        e = pb[(pb["scenario"]==scn)&(pb["period"]=="early")].set_index(merge_on)["cl_Gm3"]
        l = pb[(pb["scenario"]==scn)&(pb["period"]=="late")].set_index(merge_on)["cl_Gm3"]
        c = e.index.intersection(l.index)
        vals = (l.loc[c]/(e.loc[c]+1e-12)).values

        med  = np.median(vals)
        q25, q75 = np.percentile(vals, [25, 75])
        ax.plot([i, i], [q25, q75], color=SSP_COLORS[scn], lw=2.0,
                solid_capstyle="round", zorder=3)
        ax.plot(i, med, "o", color=SSP_COLORS[scn], ms=7,
                markeredgecolor="black", markeredgewidth=0.6, zorder=4)
        ax.text(i, q75+0.04, f"{med:.2f}", ha="center", fontsize=5.5,
                fontweight="bold")
        print(f"  (b) {SSP_LABELS[scn]}: {med:.2f}")

    ax.axhline(1.0, color="#777", ls="--", lw=0.6, zorder=0)
    ax.set_xticks(range(len(SCENARIOS)))
    ax.set_xticklabels([SSP_LABELS[s] for s in SCENARIOS], fontsize=5.5)
    ax.set_ylabel("Late / early century\nexceedance ratio")
    ax.set_xlim(-0.5, len(SCENARIOS)-0.5)
else:
    ax.text(0.5,0.5,"Daily data\nnot available",transform=ax.transAxes,
            ha="center",fontsize=7)
lgrid(ax); plabel(ax,"b")

# ── (c) Ranking stability — grouped dot plot ─────────────────────
ax = axes[2]
TOP_FRAC = 0.10
rows_rank = []
for (gcm,hydro,scn),g in df_all.groupby(["gcm","hydro","scenario"]):
    R = sorted(g["R_weight"].unique())
    if len(R)<6: continue
    Rlo, Rmi = R[3], R[len(R)//2]
    gl = g[g["R_weight"]==Rlo].sort_values("reach_id")
    gm = g[g["R_weight"]==Rmi].sort_values("reach_id")
    c = set(gl["reach_id"])&set(gm["reach_id"])
    if len(c)<50: continue
    gl_c = gl[gl["reach_id"].isin(c)]
    gm_c = gm[gm["reach_id"].isin(c)]
    rho,_ = spearmanr(gl_c["eff_total_L1"].values,
                      gm_c["eff_total_L1"].values)
    n = min(len(gl_c),len(gm_c))
    k = max(1,int(TOP_FRAC*n))
    topA = set(gl_c.nlargest(k,"eff_total_L1")["reach_id"])
    topB = set(gm_c.nlargest(k,"eff_total_L1")["reach_id"])
    ov = len(topA&topB)/k
    rows_rank.append({"scenario":int(scn),"spearman":rho,"overlap":ov})

if rows_rank:
    C = pd.DataFrame(rows_rank)
    offset = 0.12
    for i, scn in enumerate(SCENARIOS):
        cs = C[C["scenario"]==scn]
        for j, (metric, color, marker, label) in enumerate([
            ("spearman", "#4292c6", "o", r"Spearman $\rho$"),
            ("overlap",  "#E6550D", "s", "Top-10% overlap"),
        ]):
            vals = cs[metric].values
            med  = np.median(vals)
            q25, q75 = np.percentile(vals, [25, 75])
            xpos = i + (j-0.5)*offset*2

            ax.plot([xpos, xpos], [q25, q75], color=color, lw=1.5,
                    solid_capstyle="round", zorder=3)
            ax.plot(xpos, med, marker, color=color, ms=6,
                    markeredgecolor="black", markeredgewidth=0.5,
                    zorder=4, label=label if i==0 else "")

    all_v = np.concatenate([C["spearman"].values, C["overlap"].values])
    ylo = max(0, np.min(all_v)-0.05)
    ax.set_ylim(ylo, 1.02)

    ax.set_xticks(range(len(SCENARIOS)))
    ax.set_xticklabels([SSP_LABELS[s] for s in SCENARIOS], fontsize=5.5)
    ax.set_ylabel("Ranking stability")
    ax.set_xlim(-0.5, len(SCENARIOS)-0.5)
    ax.legend(loc="lower left", fontsize=5, handlelength=1.0)
    print(f"  (c) ρ={C['spearman'].median():.3f}, "
          f"overlap={C['overlap'].median():.2f}")

lgrid(ax); plabel(ax,"c")

for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS4_robustness.{ext}")
plt.show()


In [ ]:
# %%
# ═══════════════════════════════════════════════════════════════════
# FIG S5 — Planning Resolution Flexibility (3-panel maps only)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FIG S5 — Planning Resolution Flexibility")
print("=" * 60)

# ── Load HUC comparison data ─────────────────────────────────────
HUC_COMPACT  = "data/huc_spatial_compact.csv"  # UPDATE: path to HUC spatial comparison output
HUC_MAPPING  = "data/comid_huc_mapping_correct.csv"  # UPDATE: path to COMID-HUC mapping

WBD_PATHS = {
    "HUC12": "shape/WBDHU12_Brazos.shp",
    "HUC10": "data/WBD_Shape/WBDHU10.shp",  # UPDATE: path to WBD HUC10 shapefile,
    "HUC8":  "data/WBD_Shape/WBDHU8.shp",  # UPDATE: path to WBD HUC8 shapefile,
}

df_huc = pd.read_csv(HUC_COMPACT)
huc_map = pd.read_csv(HUC_MAPPING)
huc_map["COMID"] = huc_map["COMID"].astype(int)

# Fix HUC codes
df_huc = df_huc.drop(columns=["HUC8","HUC10","HUC12"], errors="ignore")
df_huc = df_huc.merge(huc_map[["COMID","HUC8","HUC10","HUC12"]],
                       left_on="reach_id", right_on="COMID",
                       how="left").drop(columns=["COMID"], errors="ignore")
for col in ["HUC8","HUC10","HUC12"]:
    df_huc[col] = df_huc[col].dropna().astype(float).astype(np.int64).astype(str)

brazos_hucs = {lv: set(df_huc[lv].dropna().unique()) for lv in ["HUC8","HUC10","HUC12"]}

# Difference columns
for lv in ["HUC12","HUC10","HUC8"]:
    df_huc[f"diff_{lv}"] = df_huc["peak_red_reach"] - df_huc[f"peak_red_{lv}"]

# Merge with network
gdf_huc = gdf_net.merge(df_huc, left_on="COMID", right_on="reach_id", how="inner")
gdf_huc = gdf_huc.sort_values("TotDASqKM_x", ascending=True)

# Load WBD
wbd_huc = {}
for lv, path in WBD_PATHS.items():
    try:
        w = gpd.read_file(path)
        for c in w.columns:
            if c.upper() == lv.upper():
                w = w.rename(columns={c: lv}); break
        w[lv] = w[lv].astype(str).str.strip()
        w = w[w[lv].isin(brazos_hucs[lv])].copy()
        w = w[~w.geometry.is_empty & w.geometry.notna()].copy()
        if gdf_net.crs is not None and w.crs is not None and w.crs != gdf_net.crs:
            w = w.to_crs(gdf_net.crs)
        wbd_huc[lv] = w
        print(f"  WBD {lv}: {len(w)} polygons")
    except Exception as e:
        print(f"  WBD {lv}: FAILED — {e}")
        wbd_huc[lv] = None

# ── Map extent ───────────────────────────────────────────────────
_bnds = gdf_net.total_bounds
_xp = (_bnds[2]-_bnds[0])*0.02; _yp = (_bnds[3]-_bnds[1])*0.02
_mxlim = (_bnds[0]-_xp, _bnds[2]+_xp)
_mylim = (_bnds[1]-_yp, _bnds[3]+_yp)

# ── FIGURE: 1×3, Fig 5 style ────────────────────────────────────
fig_s = plt.figure(figsize=(W_FULL, 85*MM))
gs_s = fig_s.add_gridspec(1, 3, left=0.01, right=0.89, top=0.92, bottom=0.14,
                           wspace=0.03)

cmap_diff = tcmap("RdBu_r", 0.05, 0.95)
diff_lim = 0.50
norm_diff = mcolors.TwoSlopeNorm(vcenter=0, vmin=-diff_lim, vmax=diff_lim)

_panels = [
    ("diff_HUC12", "a", "HUC12 (208 outlets)", "HUC12", 0.3, "99.9%"),
    ("diff_HUC10", "b", "HUC10 (42 outlets)",  "HUC10", 0.6, "98.4%"),
    ("diff_HUC8",  "c", "HUC8 (28 outlets)",   "HUC8",  1.0, "88.6%"),
]

for i, (col, lab, subtitle, bnd_lv, bnd_lw, pct) in enumerate(_panels):
    ax = fig_s.add_subplot(gs_s[0, i])
    ax.set_aspect("equal"); ax.axis("off")
    add_bg(ax, gdf_net)

    gdf_p = gdf_huc.sort_values(col, key=lambda s: s.abs(), ascending=True)
    lw_p = np.full(len(gdf_p), 0.95)
    draw_net_cmap(ax, gdf_p, gdf_p[col].values, cmap_diff, norm_diff, lw_p)

    # WBD boundary
    w = wbd_huc.get(bnd_lv)
    if w is not None and len(w) > 0:
        w.boundary.plot(ax=ax, color="#555555", linewidth=bnd_lw, alpha=0.45, zorder=5)

    ax.set_xlim(_mxlim); ax.set_ylim(_mylim)

    # Panel label
    ax.text(-0.05, 0.97, lab, transform=ax.transAxes,
            fontweight="bold", fontsize=11, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.9))

    # Subtitle below map
    ax.text(0.50, 0.02, subtitle, transform=ax.transAxes, ha="center",
            fontsize=7, fontweight="bold", color="#444",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9))

    # Performance badge
    ax.text(0.50, -0.04, f"{pct} of reach-level baseline", transform=ax.transAxes,
            ha="center", fontsize=6.5, color="#666", style="italic")

    if i == 0: add_scalebar(ax, gdf_net)

# Colorbar — right side vertical, Fig 5 style
cax_s = fig_s.add_axes([0.91, 0.18, 0.013, 0.65])
sm_s = cm.ScalarMappable(cmap=cmap_diff, norm=norm_diff)
cb_s = fig_s.colorbar(sm_s, cax=cax_s)
cb_s.set_label(r"$\Delta$ peak reduction vs. reach-level (pp)", fontsize=7, labelpad=8)
_dt = [-diff_lim, -diff_lim/2, 0, diff_lim/2, diff_lim]
cb_s.set_ticks(_dt)
cb_s.set_ticklabels([f"{t*100:+.0f}" for t in _dt])
cb_s.ax.tick_params(labelsize=6.5, width=0.3, length=2)
cb_s.outline.set_linewidth(0.3)

for ext in ("png","pdf"): fig_s.savefig(FIG_DIR / f"figS5_huc_flexibility.{ext}")
plt.show()
print("  ✓ Saved figS5_huc_flexibility")

In [ ]:
# FIG S6 — Physical Feasibility (1×3)
#   (a) Attenuation fraction histogram
#   (b) Basin-scale attenuation fraction per SSP (dot-whisker)
#   (c) |u_i|/x_i during exceedance per SSP (median + P95)
# ═════════════════════════════════════════════════════════════════
print("\nFIG S6 — Physical Feasibility")
has_af = "atten_fraction" in df_base.columns

fig, axes = plt.subplots(1, 3, figsize=(W_FULL, 70*MM),
                          gridspec_kw={"wspace":0.36})

# ── (a) Histogram — pooled across all SSPs ───────────────────────
ax = axes[0]
if has_af:
    v = df_base["atten_fraction"].dropna().values
    v = v[np.isfinite(v)&(v>=0)&(v<=1)]*100

    ax.hist(v, bins=50, density=True, color="#9ecae1", alpha=0.85,
            edgecolor="#4292c6", linewidth=0.3, zorder=2)

    med, p95 = np.median(v), np.percentile(v,95)

    ax.axvline(med, color="k", ls="--", lw=0.9, zorder=5)
    ax.axvline(p95, color="k", ls=":",  lw=0.7, zorder=5)
    yl = ax.get_ylim()[1]
    ax.text(med-1, yl*0.93, f"median\n{med:.0f}%", fontsize=6,
            fontweight="bold", ha="right", va="top")
    ax.text(p95+1, yl*0.80, f"P95\n{p95:.0f}%", fontsize=6,
            ha="left", va="top")

    ax.set_xlabel(r"Attenuation fraction $|u_i^-|/|u_i|$ (%)")
    ax.set_ylabel("Density"); ax.set_xlim(0,100)
    print(f"  (a) median={med:.1f}%, P95={p95:.1f}%")
lgrid(ax); plabel(ax,"a")

# ── (b) Basin-scale attenuation: dot + IQR whisker per SSP ──────
ax = axes[1]
if "eff_atten_L1" in df_base.columns:
    rows = []
    for (gcm,hydro,scn),g in df_base.groupby(["gcm","hydro","scenario"]):
        t = g["eff_total_L1"].sum()
        a = g["eff_atten_L1"].sum()
        rows.append({"scenario":int(scn), "val":a/(t+1e-12)*100})
    dfc = pd.DataFrame(rows)

    for i, scn in enumerate(SCENARIOS):
        vals = dfc[dfc["scenario"]==scn]["val"].values
        med  = np.median(vals)
        q25, q75 = np.percentile(vals, [25, 75])

        ax.plot([i, i], [q25, q75], color=SSP_COLORS[scn], lw=2.0,
                solid_capstyle="round", zorder=3)
        ax.plot(i, med, "o", color=SSP_COLORS[scn], ms=7,
                markeredgecolor="black", markeredgewidth=0.6, zorder=4)
        ax.text(i, q75+0.5, f"{med:.1f}%", ha="center", fontsize=5.5,
                fontweight="bold")
        print(f"  (b) {SSP_LABELS[scn]}: {med:.1f}% [{q25:.1f}–{q75:.1f}]")

    ax.set_xticks(range(len(SCENARIOS)))
    ax.set_xticklabels([SSP_LABELS[s] for s in SCENARIOS], fontsize=6)
    ax.set_ylabel("Basin-scale attenuation\nfraction (%)")
    ax.set_xlim(-0.5, len(SCENARIOS)-0.5)

    all_v = dfc["val"].values
    ax.set_ylim(np.min(all_v)-2, np.max(all_v)+3)
lgrid(ax); plabel(ax,"b")

# ── (c) |u_i|/x_i during exceedance per SSP (median + P95) ──────
ax = axes[2]
MASS_CSV = Path("mass_conservation_summary.csv")
if MASS_CSV.exists():
    df_feas = pd.read_csv(MASS_CSV)
    feas_scenarios = sorted(df_feas["ssp"].unique())

    for i, scn in enumerate(feas_scenarios):
        sub_f = df_feas[df_feas["ssp"]==scn]
        med_vals = sub_f["feas_median_mean_ratio_flow"].dropna().values * 100
        p95_vals = sub_f["feas_p95_mean_ratio_flow"].dropna().values * 100

        if len(med_vals) == 0: continue

        med_m = np.median(med_vals)
        med_p = np.median(p95_vals)
        q25_m, q75_m = np.percentile(med_vals, [25, 75])
        q25_p, q75_p = np.percentile(p95_vals, [25, 75])

        offset = 0.15
        xm = i - offset
        ax.plot([xm, xm], [q25_m, q75_m], color=SSP_COLORS[scn], lw=1.5,
                solid_capstyle="round", zorder=3)
        ax.plot(xm, med_m, "o", color=SSP_COLORS[scn], ms=6,
                markeredgecolor="black", markeredgewidth=0.5, zorder=4,
                label="Median across reaches" if i==0 else "")

        xp = i + offset
        ax.plot([xp, xp], [q25_p, q75_p], color=SSP_COLORS[scn], lw=1.5,
                solid_capstyle="round", zorder=3)
        ax.plot(xp, med_p, "s", color=SSP_COLORS[scn], ms=5.5,
                markeredgecolor="black", markeredgewidth=0.5, zorder=4,
                label="P95 across reaches" if i==0 else "")

        print(f"  (c) {SSP_LABELS[scn]}: median={med_m:.1f}%, P95={med_p:.1f}%")

    all_med_f = df_feas["feas_median_mean_ratio_flow"].dropna().values * 100
    all_p95_f = df_feas["feas_p95_mean_ratio_flow"].dropna().values * 100
    _lbl = "all: median=%d%%" % int(np.median(all_med_f)) + "\n P95=%d%%" % int(np.median(all_p95_f))
    ax.text(0.97, 0.95, _lbl,
            transform=ax.transAxes, fontsize=5.5, va="top", ha="right",
            )

    ax.set_xticks(range(len(feas_scenarios)))
    ax.set_xticklabels([SSP_LABELS[s] for s in feas_scenarios], fontsize=6)
    ax.set_ylabel(r"$|u_i|/x_i$ during exceedance (%)")
    ax.set_xlim(-0.5, len(feas_scenarios)-0.5)
    ax.legend(loc="upper left", fontsize=5, handlelength=1.0)
else:
    ax.text(0.5, 0.5, "Run si_mass_conservation.py\nto generate data",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=6.5)
    ax.set_ylabel(r"$|u_i|/x_i$ during exceedance (%)")
for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS6_feasibility.{ext}")
plt.show()


# %% ═════════════════════════════════════════════════════════════